<a href="https://colab.research.google.com/github/Ryuga-17/BeaconX-ML/blob/main/train_yolo_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PCB Defect Detection: YOLO11 Training on Google Colab

This notebook provides a complete pipeline to train a **YOLO11** model on your custom PCB defects dataset using Google Colab's GPU acceleration. It mounts Google Drive, extracts the zipped dataset, updates the dataset configuration (`data.yaml`), trains the model with PCB-specific augmentations, and saves the weights/results back to Google Drive.

### **Prerequisites & Setup:**
1. Zip your `merged_pcb` dataset directory. The zip file should contain `data.yaml` and the `images/` directory at the root level.
2. Upload your zipped dataset (e.g. `merged_pcb.zip`) to your **Google Drive** root folder or a specific directory.
3. Set Google Colab runtime to **GPU** (`Runtime` -> `Change runtime type` -> select `T4 GPU` or better).

## Step 1: Mount Google Drive
Mount your Google Drive to access the uploaded dataset and save the training results.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 2: Check GPU Acceleration
Verify that a compatible GPU (e.g., NVIDIA T4, L4, or A100) is active.

In [ ]:
!nvidia-smi

import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))

/bin/bash: line 1: nvidia-smi: command not found
CUDA Available: False


## Step 3: Install Ultralytics
Install the Ultralytics library which contains the training and inference pipeline for YOLO11.

In [ ]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 65.4 MB/s eta 0:00:00


## Step 4: Extract the Dataset
Extract your dataset zip file from Google Drive to the local high-speed directory on Colab (`/content/merged_pcb`). Using the local disk ensures maximum dataloading performance during training.

In [ ]:
import os
import zipfile

# Update these paths if your zip file is named differently or in a subfolder
zip_path_in_drive = '/content/drive/MyDrive/merged_pcb.zip'
extract_path = '/content/merged_pcb'

if not os.path.exists(zip_path_in_drive):
    print(f"❌ ERROR: Could not find '{zip_path_in_drive}' in Google Drive.")
    print("Please verify that the file name is correct and that it's located in the main MyDrive folder.")
else:
    print(f"📦 Unzipping {zip_path_in_drive} to {extract_path}...")
    os.makedirs(extract_path, exist_ok=True)
    with zipfile.ZipFile(zip_path_in_drive, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("✅ Unzipping complete!")

    # List extracted files to verify
    print("Contents of extraction directory:", os.listdir(extract_path))

📦 Unzipping /content/drive/MyDrive/merged_pcb.zip to /content/merged_pcb...
✅ Unzipping complete!
Contents of extraction directory: ['__MACOSX', 'merged_pcb']


## Step 5: Adjust `data.yaml` Paths
Your local `data.yaml` has paths configured for your local machine. We must update the `path:` variable in `data.yaml` so that YOLO knows where to find the images in the Google Colab environment.

In [ ]:
import yaml
import glob

# Search recursively for data.yaml in the extracted folder
yaml_files = glob.glob(os.path.join(extract_path, '**/data.yaml'), recursive=True)

if yaml_files:
    data_yaml_path = yaml_files[0]
    dataset_root = os.path.dirname(data_yaml_path)

    with open(data_yaml_path, 'r') as f:
        data = yaml.safe_load(f)

    # Modify the dataset root path key to point to the actual folder containing data.yaml on Colab
    old_path = data.get('path', 'Not Specified')
    data['path'] = dataset_root

    print(f"🔍 Found data.yaml at: {data_yaml_path}")
    print(f"🔄 Updating path in data.yaml:")
    print(f"   From: {old_path}")
    print(f"   To:   {data['path']}")

    # Save modified data.yaml
    with open(data_yaml_path, 'w') as f:
        yaml.safe_dump(data, f, default_flow_style=False)

    print("✅ data.yaml successfully patched for Colab!")

    # Print configuration content for confirmation
    print("\n--- Current data.yaml Content ---")
    with open(data_yaml_path, 'r') as f:
        print(f.read())
else:
    print(f"❌ ERROR: data.yaml was not found under {extract_path}. Please verify the directory structure of the extracted zip file.")

🔍 Found data.yaml at: /content/merged_pcb/merged_pcb/data.yaml
🔄 Updating path in data.yaml:
   From: /Users/viveksawant/Desktop/tredence/merged_pcb
   To:   /content/merged_pcb/merged_pcb
✅ data.yaml successfully patched for Colab!

--- Current data.yaml Content ---
names:
  0: missing_hole
  1: mouse_bite
  2: open_circuit
  3: short
  4: spur
  5: spurious_copper
path: /content/merged_pcb/merged_pcb
test: images/test
train: images/train
val: images/val



## Step 6: Train YOLO11 Model
Now we initialize the training run.
- We use **YOLO11 Medium** (`yolo11m.pt`) as our base architecture.
- We direct the `project` argument to a folder in your Google Drive (`/content/drive/MyDrive/pcb_defects_training`) so checkpoints are written directly to the cloud. If your connection drops, your progress and model weights (`best.pt`, `last.pt`) will remain safe.
- We use specific augmentations tailored for high-accuracy PCB defect detection (e.g., full rotations, scaling, mixup, and mosaic).

In [ ]:
from ultralytics import YOLO

# 1. Initialize YOLO11 model (choose 'yolo11n.pt', 'yolo11s.pt', 'yolo11m.pt', 'yolo11l.pt', or 'yolo11x.pt')
model_name = 'yolo11m.pt'
print(f"🚀 Initializing {model_name}...")
model = YOLO(model_name)

# 2. Define target directory in Google Drive to preserve weights and runs
project_drive_path = '/content/drive/MyDrive/pcb_defects_training'
run_name = 'yolo11m_imgsz1280_colab'

# 3. Run training using the dynamically located data_yaml_path
results = model.train(
    data=data_yaml_path,
    epochs=10,               # Set total epochs (e.g., 50-100 is typical for initial runs)
    imgsz=1280,               # High-resolution input matching the detail level needed for PCB flaws
    batch=-1,                 # Auto-batch size (finds optimal batch size for Colab's GPU VRAM)
    device=0,                 # Train on GPU
    project=project_drive_path,
    name=run_name,
    exist_ok=True,
    patience=4,

    # --- PCB Specific Augmentations ---
    degrees=180.0,            # Support full rotation of boards
    scale=0.5,                # Scale jitter for different distance/height views
    fliplr=0.5,               # Random horizontal flip
    flipud=0.5,               # Random vertical flip
    mosaic=1.0,               # Combine multiple images to improve context
    mixup=0.15,               # Mix image overlaps for regularization
    close_mosaic=10,          # Disable mosaic for last 10 epochs for fine-tuning

    # --- Logging & Performance ---
    save=True,                # Save model weights every epoch
    plots=True,               # Save evaluation curves and plots
    workers=8,                # Multi-threaded data loading
)

print("\n🎉 Training session complete!")
print(f"Checkpoints and reports are stored at: {project_drive_path}/{run_name}")

🚀 Initializing yolo11m.pt...
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/merged_pcb/merged_pcb/data.yaml, degrees=180.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11m_imgsz1280_colab, nbs=64, nms=False, opset=None, optimi

In [ ]:
from ultralytics import YOLO

# 1. Initialize YOLO11 model (choose 'yolo11n.pt', 'yolo11s.pt', 'yolo11m.pt', 'yolo11l.pt', or 'yolo11x.pt')
model_name = 'yolo11m.pt'
print(f"🚀 Initializing {model_name}...")
model = YOLO(model_name)
run_name = 'yolo11m_imgsz1280_colab'

project_drive_path = '/content/drive/MyDrive/pcb_defects_training'
last_checkpoint_path = os.path.join(project_drive_path, run_name, 'weights', 'last.pt')

if os.path.exists(last_checkpoint_path):
    print(f"🔄 Found checkpoint at {last_checkpoint_path}. Resuming training...")
    resume_model = YOLO(last_checkpoint_path)
    # Resume training using the last checkpoint's training state
    resume_model.train(resume=True)
else:
    print(f"❌ Checkpoint not found at {last_checkpoint_path}.")

🚀 Initializing yolo11m.pt...
🔄 Found checkpoint at /content/drive/MyDrive/pcb_defects_training/yolo11m_imgsz1280_colab/weights/last.pt. Resuming training...
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cpu 


ValueError: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 0
os.environ['CUDA_VISIBLE_DEVICES']: 0
See https://pytorch.org/get-started/locally/ for up-to-date torch install instructions if no CUDA devices are seen by torch.


In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


False
0
No GPU


## Step 7: Plot Results
Load and display the performance curves (`results.png`) generated during the training session.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

results_dir = os.path.join(project_drive_path, run_name)
results_png = os.path.join(results_dir, 'results.png')

if os.path.exists(results_png):
    plt.figure(figsize=(15, 12))
    img = mpimg.imread(results_png)
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print("📈 The results plot was not found yet. Verify that training completed or check your Drive path.")